# RootCause SDK quickstart

Causal discovery, digital twins, and what-if questions from Python. Two modes, one object model:

- **Direct mode** — `rc.discover(df)` on a pandas DataFrame, zero workspace ceremony.
- **Platform mode** — `rc.workspace(...)` over everything your team builds in the RootCause UI.

```bash
pip install rootcause-sdk
```

In [ ]:
import os
import numpy as np
import pandas as pd

import rootcause as rc

rc.login(base_url=os.environ.get("ROOTCAUSE_BASE_URL", "https://platform.rootcause.ai"))
rc.workspaces()

## Direct mode: from DataFrame to causal graph

A synthetic marketing funnel with known ground truth: `marketing_spend -> leads -> revenue`,
with `seasonality` nudging leads. Discovery should recover exactly that structure.

In [ ]:
rng = np.random.default_rng(7)
n = 240
marketing = rng.normal(50, 12, n)
seasonality = rng.normal(0, 1, n)
leads = 3.0 * marketing + 15 * seasonality + rng.normal(0, 8, n)
revenue = 2.2 * leads + rng.normal(0, 20, n)

df = pd.DataFrame({
    "marketing_spend": marketing.round(2),
    "seasonality": seasonality.round(3),
    "leads": leads.round(1),
    "revenue": revenue.round(1),
})
df.head()

In [ ]:
graph = rc.discover(df)
graph.edges

The adjacency matrix is a labelled DataFrame — `.to_numpy()` and `.to_networkx()` are there when you need them.

In [ ]:
graph.adjacency()

## Domain knowledge, then training

`pin` fixes an edge as present; `forbid` as absent. Training fits the causal Bayesian network.

In [ ]:
graph.pin("marketing_spend", "leads")
twin = graph.train()
twin

## The power-user primitive: raw joint draws

Every simulation family wraps conditional sampling. `twin.sample()` hands you the draws so you can
compute your own estimands. Seeds are reproducible across every twin family.

In [ ]:
draws = twin.sample(n=2000, seed=42)
draws.to_frame().describe().round(1)

In [ ]:
boosted = twin.sample(n=2000, do={"marketing_spend": rc.pct(+20)}, seed=42)
pd.DataFrame({
    "baseline": draws.to_frame().mean(),
    "do(marketing +20%)": boosted.to_frame().mean(),
}).round(1)

## Interventions, narrated

`intervene` runs the full simulation machinery server-side and blocks for the result.

In [ ]:
result = twin.intervene({"marketing_spend": rc.pct(+25)}, outcomes=["revenue", "leads"])
result

## Ontology queries

Every upload gets ontology concepts. The query engine handles filters, aggregations, grouping,
and natural language — `select` takes concept names.

In [ ]:
from rootcause.direct import scratch_workspace
onto = scratch_workspace(rc._transport()).ontology
onto.concepts

In [ ]:
onto.query(select=["Revenue", "Leads"], limit=5).to_frame(max_rows=5)

## Portable twins

The export zip carries the trained model parameters; `rc.load_twin` brings it back anywhere.

In [ ]:
path = twin.save("quickstart.rctwin")
f"{path.name}: {path.stat().st_size:,} bytes"

## Platform mode

The same classes over shared workspaces — twins your colleagues trained in the UI are just there:

```python
ws = rc.workspace("Calix Forecasting")
ws.sources["shipments"].to_frame()

twin = ws.twin("C8 Temporal")
fc = twin.forecast(horizon=24)
fc.to_frame()

twin.ask("what happens to bookings if we cut trade shows entirely?")
```